# 19 - Word2Vec + RNN with Truncated BPPT & Gradient Clipping

Goal: Fix vanishing/exploding gradients problem from Notebook 18 using:
1. **Truncated BPPT** - Only backprop through last N steps (not all 128)
2. **Gradient Clipping** - Limit gradient magnitude during updates
3. **LeakyReLU option** - Alternative to tanh with better gradient flow

Problem analyzed in Notebook 18:
- RNN Vanilla suffers from vanishing/exploding gradients
- Gradients multiplied by Wh for each timestep: multiplied 128+ times
- If derivada < 1, gradients → 0 (vanishing); if > 1, gradients → ∞ (exploding)

Run with: conda activate tfenv

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Cargar embeddings Word2Vec preentrenados
class Word2VecLoader:
    def __init__(self, path='myWord2Vec/v2/'):
        target_embeddings = np.load(path + 'target_embeddings.npy')
        context_embeddings = np.load(path + 'context_embeddings.npy')
        text_vocab = np.load(path + 'text_vocab.npy', allow_pickle=True).item()

        self.target_embeddings = target_embeddings
        self.context_embeddings = context_embeddings
        self.final_embeddings = (target_embeddings + context_embeddings) / 2
        self.text_vocab = text_vocab
        self.idx_to_word = {idx: word for word, idx in text_vocab.items()}
        self.embedding_dim = target_embeddings.shape[1]

        self.embedding_layer = layers.Embedding(
            input_dim=target_embeddings.shape[0],
            output_dim=target_embeddings.shape[1],
            weights=[target_embeddings],
            trainable=False,
            name='pretrained_embedding'
        )

        print('Embeddings cargados:', target_embeddings.shape, context_embeddings.shape)
        print('Vocabulario cargado:', len(text_vocab))

    def encode(self, words):
        return [self.text_vocab[w] for w in words if w in self.text_vocab]

    def decode(self, token_id):
        return self.idx_to_word.get(int(token_id), '<unk>')

loader = Word2VecLoader()

In [ ]:
# TRUNCATED BPPT: Usar ventana más larga (128 en lugar de 5)
# Pero solo hacer backprop en últimos 50 pasos
print('Loading gaianet/london dataset...')
ds = load_dataset('gaianet/london', split='train')
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:50000]
full_text = ' '.join(texts[:50000])

words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2 and w in loader.text_vocab]
print(f'Total words used: {len(words)}')
print(f'Sample: {words[:20]}')

def create_sequences(words, window=128, step=10):
    """Crear secuencias largas para Truncated BPPT
    window=128: contexto largo para mejor representación
    step=10: saltar cada 10 palabras (reducir datos sin perder información)
    """
    X, y = [], []
    for i in range(0, len(words) - window, step):
        context = words[i:i + window]
        target = words[i + window]
        if all(w in loader.text_vocab for w in context + [target]):
            X.append([loader.text_vocab[w] for w in context])
            y.append(loader.text_vocab[target])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X, y = create_sequences(words, window=128)
print(f'Sequences created: {len(X)}')
print(f'Input shape: {X.shape}, Target shape: {y.shape}')

if len(X) == 0:
    raise ValueError('No se pudieron crear secuencias con el vocabulario cargado.')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
# RNN con Truncated BPPT, Gradient Clipping, y opción LeakyReLU
# Soluciones:
# 1. Truncated BPPT: Solo hacer backprop en últimos N pasos (tf.stop_gradient en pasos iniciales)
# 2. Gradient Clipping: clip_norm limita magnitud de gradientes
# 3. LeakyReLU: Alternativa a tanh con mejor gradient flow (derivada siempre > 0)

class ManualRNNWordPredictorV2(keras.Model):
    def __init__(self, embedding_layer, vocab_size, embed_dim, hidden_dim=128, use_leaky_relu=False, truncate_steps=50):
        super().__init__()
        self.embedding = embedding_layer
        self.hidden_dim = hidden_dim
        self.use_leaky_relu = use_leaky_relu
        self.truncate_steps = truncate_steps  # Solo backprop en últimos N pasos
        
        self.Wx = layers.Dense(hidden_dim, use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(0.0001))
        self.Wh = layers.Dense(hidden_dim, use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(0.0001))
        self.b = self.add_weight(shape=(hidden_dim,), initializer='zeros', trainable=True, name='bias')
        self.out = layers.Dense(vocab_size, activation='softmax')

    def call(self, inputs):
        x = self.embedding(inputs)  # (batch, 128, 64)
        x = tf.nn.dropout(x, rate=0.1)
        h = tf.zeros((tf.shape(x)[0], self.hidden_dim))
        
        x_unstacked = tf.unstack(x, axis=1)
        sequence_length = len(x_unstacked)
        
        # Recorrer timesteps con TRUNCATED BPPT
        for i, xt in enumerate(x_unstacked):
            if self.use_leaky_relu:
                h = tf.nn.leaky_relu(self.Wx(xt) + self.Wh(h) + self.b, alpha=0.2)
            else:
                h = tf.tanh(self.Wx(xt) + self.Wh(h) + self.b)
            
            # TRUNCATED BPPT: Detener gradientes en pasos iniciales
            # Solo permite backprop en últimos truncate_steps pasos
            if i < sequence_length - self.truncate_steps:
                h = tf.stop_gradient(h)  # Detiene backpropagation aquí
        
        return self.out(h)

vocab_size = loader.target_embeddings.shape[0]

# Probar primero SIN LeakyReLU (tanh normal con gradient clipping + truncated BPPT)
model = ManualRNNWordPredictorV2(
    loader.embedding_layer, vocab_size, loader.embedding_dim, 
    hidden_dim=64, use_leaky_relu=False, truncate_steps=50
)

# SOLUCIONES COMBINADAS:
# 1. Gradient Clipping - Limita magnitud de gradientes a 5.0
# 2. Truncated BPPT - Solo backprop en últimos 50 pasos (de 128)
optimizer = keras.optimizers.Adam(learning_rate=0.001, clipnorm=5.0)

model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.build((None, 128))
model.summary()

batch_size = 32  # Reducido porque secuencias son más largas
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(min(len(X_train), 1000)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

callbacks=[keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=5, 
    restore_best_weights=True
)]

history = model.fit(train_ds, validation_data=val_ds, epochs=100, verbose=1, callbacks=callbacks)

In [ ]:
# Evaluación y predicción de siguiente palabra
from sklearn.metrics import accuracy_score

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy (tanh + Gradient Clipping + Truncated BPPT): {acc:.3f}')

def predict_next_word(context_words, top_n=5, use_last_n=128):
    context_ids = [loader.text_vocab[w] for w in context_words if w in loader.text_vocab]
    # Rellenar con palabras random si no tenemos suficientes
    if len(context_ids) < use_last_n:
        random_ids = np.random.randint(0, vocab_size, use_last_n - len(context_ids))
        context_ids = list(random_ids) + context_ids
    context_ids = np.array([context_ids[-use_last_n:]], dtype=np.int32)
    probs = model.predict(context_ids, verbose=0)[0]
    top_indices = np.argsort(probs)[-top_n:][::-1]
    return [(loader.decode(idx), float(probs[idx])) for idx in top_indices]

sample_contexts = [
    ['london', 'bridge', 'river', 'city', 'center'],
    ['bank', 'of', 'england', 'is', 'located'],
    ['queen', 'of', 'england', 'lives', 'in']
]

for context in sample_contexts:
    preds = predict_next_word(context, top_n=5)
    if preds:
        print(f"Contexto {context} -> {preds}")

plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('RNN with Truncated BPPT + Gradient Clipping')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# OPCIÓN 2: Si tanh + clipping no es suficiente, probar LeakyReLU
# LeakyReLU tiene derivada > 0 siempre (no desvanece como tanh)

print("\n=== Intentando con LeakyReLU en lugar de tanh ===")
model_leaky = ManualRNNWordPredictorV2(
    loader.embedding_layer, vocab_size, loader.embedding_dim, 
    hidden_dim=64, use_leaky_relu=True, truncate_steps=50
)

optimizer_leaky = keras.optimizers.Adam(learning_rate=0.001, clipnorm=5.0)
model_leaky.compile(optimizer=optimizer_leaky, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_leaky.build((None, 128))

history_leaky = model_leaky.fit(train_ds, validation_data=val_ds, epochs=100, verbose=0, callbacks=callbacks)

loss_leaky, acc_leaky = model_leaky.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy (LeakyReLU + Gradient Clipping + Truncated BPPT): {acc_leaky:.3f}')

# Comparar ambos modelos
print(f"\nComparación:")
print(f"  tanh + Gradient Clipping + Truncated BPPT: {acc:.3f}")
print(f"  LeakyReLU + Gradient Clipping + Truncated BPPT: {acc_leaky:.3f}")

# Graficar ambos
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train (tanh)', linewidth=2)
plt.plot(history.history['val_accuracy'], label='val (tanh)', linewidth=2)
plt.title('tanh + Gradient Clipping + Truncated BPPT')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_leaky.history['accuracy'], label='train (LeakyReLU)', linewidth=2)
plt.plot(history_leaky.history['val_accuracy'], label='val (LeakyReLU)', linewidth=2)
plt.title('LeakyReLU + Gradient Clipping + Truncated BPPT')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()